### Ingesting races file

In [0]:
%run ../00-common/1.environment_config

In [0]:
%run ../00-common/2.bronze_helpers

In [0]:
dbutils.widgets.text("p_batch_id", "")  
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
source_folder = f'{landing_folder_path}/{v_batch_id}/results'
table_name = f'{catalog_name}.{bronze_schema}.results'

In [0]:
# Schema Validation
from pyspark.sql.types import StructField, StructType, StringType, LongType, DoubleType

results_schema = StructType([
    StructField('constructorId', StringType(), True),
    StructField('date', StringType(), True),
    StructField('driverId', StringType(), True),
    StructField('grid', LongType(), True),
    StructField('laps', LongType(), True),
    StructField('number', LongType(), True),
    StructField('points', DoubleType(), True),
    StructField('position', LongType(), True),
    StructField('positionText', StringType(), True),
    StructField('raceName', StringType(), True),
    StructField('round', LongType(), True),
    StructField('season', LongType(), True),
    StructField('status', StringType(), True),
    StructField('url', StringType(), True)
])

In [0]:
results_df = (
    spark.read
    .format('json')
    .option('mode','FAILFAST')
    .schema(results_schema) 
    .load(source_folder)
)

In [0]:
display(results_df)

In [0]:
# Ingesting metadata
results_df_final = add_file_metadata(results_df)

In [0]:
display(results_df_final)

In [0]:
write_to_bronze(results_df_final, table_name, v_batch_id)

In [0]:
display(spark.table(table_name))

In [0]:
%sql
SELECT season, COUNT(*)
    FROM formula1_incr.bronze.results
GROUP BY season
ORDER BY season;